# MM7 lab: layers (30 min)

**What changed since MM6.** So far one protocol did everything. Today the work is split over **layers**: each layer offers a *service* to the layer above, and talks to its *peer* layer on the other side through a *protocol*. The lab follows Problem Set 7.

| Part | Time | | Problem Set 7 |
|---|---|---|---|
| A | 12 min | Follow a packet from Zoya to Yoshi through a router | Problem 1(c) |
| B | 18 min | Reliability at two layers | Problem 2 |

**What you build:** two small functions in `student/mm7_layers.py`, with Copilot. The layer engine is given.

**Rhythm for every question:** 1. predict → 2. run → 3. observe → 4. change one knob → 5. answer and check.
Check a short answer with `check_answer("A1", your_answer)`. It only tells you *correct* or *not correct*.

**Rules.** Prediction first, then run. Copilot writes code in `student/` only.

In [ ]:
%load_ext autoreload
%autoreload 2
import sys; sys.path.insert(0, "..")   # the CommSim folder
from commsim import *
from commsim.checks import check_mm7_encapsulation, check_mm7_retry
from student.mm7_layers import packet_sizes, retry_decision

## Part A: follow a packet through the layers (12 min)

Problem 1(c): Zoya sends a packet to Yoshi through a router. Zoya and Yoshi use the 7-layer OSI model; the router only has layers 1 to 3.

> **Anchor question.** Which headers are added and removed at each stage, how far up the stack does the router process the packet, and which information is link-specific versus end-to-end?

**A0. Warm-up (Problem 1(b)).** Match each description with *service*, *protocol* or *interface*:

1. What a layer offers to the layer above it, for example "deliver this packet to Yoshi".
2. The boundary between two neighbouring layers in the same node, through which the upper layer uses the lower one.
3. The rules (message formats, ACKs, timeouts) that two peers at the same layer use to talk to each other, for example Zoya's transport layer and Yoshi's transport layer.

In [ ]:
check_answer("A0", ...)   # a list of three words, for 1, 2 and 3: ["...", "...", "..."]

**A1. Predict first:** what is the highest layer the router processes? Enter the layer number.

In [ ]:
check_answer("A1", ...)

**A2. Run.** The provided scenario: 100 B of application data, an 8 B transport header, a 12 B network header and a 10 B data link header. Follow the arrows: down Zoya's stack, over link 1, up and down the router, over link 2, up Yoshi's stack. The bars at the bottom show the packet on each link.

In [ ]:
trace = zoya_router_yoshi(payload_bytes=100, transport=8, network=12, data_link=10)
print(trace)
layer_diagram(trace);

**A3. Observe.** Use the printout and the diagram.

- **(a)** Does the router inspect the application header? (yes / no)
- **(b)** Which header is link-specific and can be replaced on the next hop? (name the layer)
- **(c)** Do both end systems use all seven layers? (yes / no)

In [ ]:
check_answer("A3a", ...)
check_answer("A3b", ...)
check_answer("A3c", ...)

**A4. Count the overhead.**

1. Ask Copilot to implement `packet_sizes` in `student/mm7_layers.py` (read the specification there first). Then run the check.

In [ ]:
check_mm7_encapsulation(packet_sizes)

2. Use your function for the numbers from A2: 100 B of application data, then 8 B transport, 12 B network and 10 B data link headers. What is the **total packet size** in bytes, and what is the **payload efficiency** (application data / total, as a fraction with two decimals)?

In [ ]:
# use packet_sizes here

In [ ]:
check_answer("A4a", ...)   # total packet size in bytes
check_answer("A4b", ...)   # payload efficiency, e.g. 0.42

**A5. Change one knob.** *Predict first:* if the headers grow while the payload stays at 100 B, does the payload efficiency rise or fall?

In [ ]:
check_answer("A5", ...)   # "rise" or "fall"

Now play: change **one** header size at a time in the cell below and rerun. For example `network=40`, add `session=4`, or give the router's outgoing link a different header with `data_link_out=20`. Watch the diagram and the efficiency.

In [ ]:
trace = zoya_router_yoshi(payload_bytes=100, transport=8, network=12, data_link=10)
print(f"packet on link 1: {trace.packet_bytes} B, payload efficiency {trace.efficiency:.3f}")
layer_diagram(trace);

**Anchor question, in your own words (2 to 3 sentences):** which header information is link-specific, which is end-to-end, and why does the router not need to look further than layer 3?

> **Your answer:** *(write here)*

## Part B: reliability at two layers (18 min)

Problem 2: Zoya's application data goes to Yoshi through a **higher layer (HL)** and a **lower layer (LL)**. Each layer can have its own ACKs, timeouts and retransmissions.

*Words:* Problem Set 7 calls these mechanisms "flow control". Here we call them **reliability** or **error control** (ACK/NACK, timeout, retransmission), because "flow control" usually means slowing a sender down to the receiver's rate.

> **Anchor question.** If both HL and LL use ACKs, timeouts and retransmissions, when does the extra mechanism improve robustness, and when does it only create extra transmissions, delay and overhead?

**The simulator** (Lecture 7, Example 1):

- Time runs in slots. A frame sent in slot *t* arrives in slot *t* + 1, so one LL attempt (DATA out, ACK back) takes **2 slots**.
- Every frame, DATA or ACK, is lost with probability PER. With a fixed `seed`, every run sees the same channel, so comparisons are fair.
- `reliability="LL"`: the LL does ARQ and offers a reliable service; the HL just hands packets down.
  `reliability="HL"`: the LL sends each frame once, without ACKs; the HL does ARQ end to end.
  `reliability="both"`: both layers do ARQ, each with its own timer.
- The HL's ACKs are ordinary data for the LL.

| Parameter | Meaning |
|---|---|
| `per` | packet error probability on the channel |
| `k_ll` | maximum number of LL retransmissions (so at most `k_ll + 1` attempts) |
| `t_ll` | LL ACK timeout in slots (2 = one attempt cycle) |
| `k_hl` | maximum number of HL retransmissions |
| `t_hl` | HL ACK timeout in slots |

**B1. Warm-up (Problem 2(a)).** There are 16 applications in the pool. How many bits does Zoya's HL need for the application identifier?

In [ ]:
check_answer("B1", ...)

**Your task: the retry rule.** Both layers decide when to retransmit with the same small rule. Ask Copilot to implement `retry_decision` in `student/mm7_layers.py` from its specification, then run the check. All experiments below use your rule.

In [ ]:
check_mm7_retry(retry_decision)

**B2. LL reliability only.** 100 packets, PER = 0.2, K_LL = 2, T_LL = 2. Note the delivered packets, the LL transmissions and retransmissions, the mean delay and the goodput. In the timeline, find a frame that the LL had to send again, and what caused it.

In [ ]:
r_ll = two_layer_link(retry_decision, reliability="LL", per=0.2, k_ll=2, t_ll=2, seed=7)
print(r_ll)
layer_timeline(r_ll, n_slots=30);

**B3. HL only, then both.** Same packets, same channel (same seed). The HL uses K_HL = 2 and T_HL = 8.

In [ ]:
r_hl = two_layer_link(retry_decision, reliability="HL", per=0.2, k_hl=2, t_hl=8, seed=7)
r_both = two_layer_link(retry_decision, reliability="both", per=0.2, k_ll=2, t_ll=2, k_hl=2, t_hl=8, seed=7)
print(r_hl, "\n")
print(r_both, "\n")
compare_runs({"LL only": r_ll, "HL only": r_hl, "HL + LL": r_both})

- **B2.** How many more LL transmissions does HL + LL need than LL only?
- **B3.** Which configuration has the most protocol overhead: LL, HL or HL + LL? (Look at the header bits and the efficiency.)

In [ ]:
check_answer("B2", ...)
check_answer("B3", ...)

**B4. The timeout trap (Problem 2(c)).** With K_LL = 2 and T_LL = 2, the LL may need up to 3 attempts × 2 slots = **6 slots** before it is done with a packet. Now set the HL timeout shorter than that: **T_HL = 4**.

*Predict first:* can the HL time out while the LL is still working on the packet? (yes / no)

In [ ]:
check_answer("B4", ...)

In [ ]:
r4 = two_layer_link(retry_decision, reliability="both", per=0.2, k_ll=2, t_ll=2, k_hl=2, t_hl=4, seed=7)
print(r4)
layer_timeline(r4, start=46, n_slots=30);   # slots 46 to 75

In the timeline (slots 46 to 75), find a **red** HL retransmission: the HL sends a new copy while its LL still holds the earlier one. What happens to the extra copy at Yoshi?

> **Your answer:** *(write here)*

**B5. Change the HL timeout.** *Predict first:* if you increase T_HL from 4 to 7, do the unnecessary HL retransmissions increase, decrease or stay unchanged?

In [ ]:
check_answer("B5", ...)   # "increase", "decrease" or "unchanged"

In [ ]:
r7 = two_layer_link(retry_decision, reliability="both", per=0.2, k_ll=2, t_ll=2, k_hl=2, t_hl=7, seed=7)
compare_runs({"T_HL = 4": r4, "T_HL = 7": r7})

**Play: change one knob at a time.** Start from the T_HL = 7 run and change **one** of `per`, `k_ll`, `t_ll`, `k_hl`, `t_hl`. Before each run, write down what you expect.

In [ ]:
# your experiments: one parameter at a time

**Anchor question, in your own words (2 to 3 sentences):** when does reliability at both layers help, and when does it only cost transmissions, delay and overhead? Think about Problem 2(b) and 2(c).

> **Your answer:** *(write here)*